# FG-CLIP text hard negative 파인튜닝 (Colab)

대조군(`HN=0`)과 text hard negative 실험(`HN=2`)을 **같은 설정**으로 학습하고, val 점수를 비교한 뒤 서버 제출용 zip까지 만든다.

**순서**: 1) Drive 마운트 → 2) 설정 → 3) 환경 설치 → 4) 데이터 압축 해제 → 5) 코드 파일 생성 → 6) 대조군 학습 → 7) HN 학습 → 8) 결과 비교 → 9) zip 생성(Drive 저장)

- 런타임 → 런타임 유형 변경에서 **GPU**를 고른다 (A100/L4 권장, T4도 가능: bf16 없으면 fp16 자동).
- 결과는 Drive(`OUT_DIR`)에 저장되므로 런타임이 끊겨도 남는다.
- val 점수는 val 500개 안에서의 **상대 비교용**이다 (후보 풀이 500장이라 test보다 쉽다). 최종 점수는 zip을 Modal L4에 올려 test로 잰다. **하이퍼파라미터는 val로만 고른다.**
- test 데이터(test_v1.tar)는 이 노트북에서 쓰지 않는다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설정

In [ ]:
import os, glob, json, subprocess

# --- 데이터 ---
TRAIN_TAR = ""        # 비우면 MyDrive 에서 train_v1.tar 를 자동 탐색
VAL_IDS_PATH = ""     # (선택) Mac 에서 make_val_ids.py 로 만든 val_ids.txt 의 Drive 경로. 비우면 무작위 val 분할

# --- 학습 ---
EPOCHS = 3
BS = 64               # OOM 이면 32
LR = 1e-5
LAST_N = 4            # 마지막 N 블록 + 투영층만 학습
HN = 2                # text hard negative 개수 (대조군은 자동으로 0 으로 별도 실행)
SEED = 0
REPO = "qihoo360/fg-clip-base"

# --- 출력 / 환경 ---
OUT_DIR = "/content/drive/MyDrive/haikuthon_runs"
TRANSFORMERS_PIN = "transformers==4.46.3"   # 서버 zip 의 requirements.txt 와 같은 버전 (원격코드 호환)

os.makedirs(OUT_DIR, exist_ok=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU 로 바꾸세요)")
print("bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None)

## 3. 환경 설치

In [ ]:
!pip install -q {TRANSFORMERS_PIN} einops
import transformers; print("transformers", transformers.__version__)

## 4. 데이터: Drive → /content 로 복사 후 압축 해제

tar 를 Drive 에서 직접 풀면 느리므로 `/content` 로 복사해서 푼다 (런타임이 초기화되면 다시 해야 함).

In [ ]:
if not TRAIN_TAR:
    hits = glob.glob("/content/drive/MyDrive/**/train_v1.tar", recursive=True)
    assert hits, "MyDrive 에서 train_v1.tar 를 못 찾음 -> TRAIN_TAR 에 경로를 직접 적으세요"
    TRAIN_TAR = hits[0]
print("TRAIN_TAR =", TRAIN_TAR)

os.makedirs("/content/data", exist_ok=True)
if not glob.glob("/content/data/**/prompts.jsonl", recursive=True):
    !cp "{TRAIN_TAR}" /content/train_v1.tar
    !tar -xf /content/train_v1.tar -C /content/data
    !rm /content/train_v1.tar

PROMPTS = glob.glob("/content/data/**/prompts.jsonl", recursive=True)[0]
IMAGES = os.path.join(os.path.dirname(PROMPTS), "images")
n_img = sum(1 for _ in glob.iglob(IMAGES + "/**/*.png", recursive=True))
print("PROMPTS =", PROMPTS); print("IMAGES  =", IMAGES, "| png:", n_img)
!head -c 600 "{PROMPTS}"

## 5. 코드 파일 생성 (이 노트북이 자체 완결되도록 파일 내용을 포함)

In [ ]:
!mkdir -p /content/work/scripts /content/work/submission/fgclip

In [ ]:
%%writefile /content/work/hn_rules.py
"""공개 train 프롬프트 텍스트만으로 만드는 규칙 기반 text hard negative (slot 하나만 바꾼 변형).

참가자도 쓸 수 있는 정보(프롬프트 문장)만 사용한다. 주최측 비공개 variants 파일은 쓰지 않는다.
종류: direction(left<->right, above<->below) / count(two<->three, three<->four; both<->all three) / attribute(색상).
"""
from __future__ import annotations

import random
import re

COLORS = ["red", "blue", "green", "yellow", "purple", "orange", "pink", "brown", "black", "white",
          "gray", "grey", "beige", "silver", "gold", "teal", "cyan", "maroon", "navy", "turquoise"]
_DIR = {"left": "right", "right": "left", "above": "below", "below": "above"}
_CNT = {"two": "three", "three": "two", "four": "three"}


def _match_case(src: str, dst: str) -> str:
    return dst.capitalize() if src[:1].isupper() else dst


def swap_direction(text: str, rng: random.Random):
    hits = list(re.finditer(r"\b(left|right|above|below)\b", text, flags=re.I))
    if not hits:
        return None
    m = rng.choice(hits)
    w = m.group(0)
    return text[:m.start()] + _match_case(w, _DIR[w.lower()]) + text[m.end():]


def swap_count(text: str, rng: random.Random):
    # "(two|three|four) <복수명사>" 형태만 (단수<->복수는 문법이 깨져 제외)
    hits = list(re.finditer(r"\b(two|three|four)\s+(?!objects\b)([a-z-]+s)\b", text, flags=re.I))
    if not hits:
        return None
    m = rng.choice(hits)
    old = m.group(1)
    new = _CNT[old.lower()]
    out = text[:m.start(1)] + _match_case(old, new) + text[m.end(1):]
    if old.lower() == "two" and new == "three":
        out = re.sub(r"\bBoth\b", "All three", out)
        out = re.sub(r"\bboth\b", "all three", out)
    elif old.lower() == "three" and new == "two":
        out = re.sub(r"\bAll three\b", "Both", out)
        out = re.sub(r"\ball three\b", "both", out)
    return out


def swap_attribute(text: str, rng: random.Random):
    pat = r"\b(" + "|".join(COLORS) + r")\b"
    hits = list(re.finditer(pat, text, flags=re.I))
    if not hits:
        return None
    m = rng.choice(hits)
    old = m.group(0)
    new = rng.choice([c for c in COLORS if c != old.lower() and c not in ("grey",) ])
    return text[:m.start()] + _match_case(old, new) + text[m.end():]


RULES = {"direction": swap_direction, "count": swap_count, "attribute": swap_attribute}


def make_hard_negatives(text: str, k: int, rng: random.Random):
    """가능한 규칙 중 최대 k개(서로 다른 종류)를 적용해 [(kind, text)] 반환. 원문과 같은 결과는 버림."""
    kinds = list(RULES)
    rng.shuffle(kinds)
    out = []
    for kind in kinds:
        new = RULES[kind](text, rng)
        if new and new != text:
            out.append((kind, new))
        if len(out) >= k:
            break
    return out

In [ ]:
%%writefile /content/work/train_fgclip_hn.py
"""FG-CLIP v1 파인튜닝: in-batch 대조학습 (+ 선택적으로 text hard negative). Colab Pro 용.

--hn 0  : 대조군 (hard negative 없음)
--hn 2  : 프롬프트마다 규칙 기반 text HN 최대 2개를 image->text 로짓의 추가 negative 로 사용 (NegCLIP 방식)

예 (Colab):
  pip install -q "transformers==4.46.3" einops
  python train_fgclip_hn.py --prompts train_prompts_v2.jsonl --images /content/train_v1/images \
      --repo qihoo360/fg-clip-base --hn 2 --out runs/hn2
결과: runs/<name>/ (safetensors + 원격코드 + 토크나이저) -> build_fgclip_submission.py --repo runs/<name> 로 zip 생성.
검증 점수는 train 내부 holdout 이라 같은 묶음(bundle) 문항이 섞여 과대평가될 수 있다. 최종 판단은 서버/테스트셋 점수로.
"""
import argparse, json, math, random, re, sys, time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(Path(__file__).resolve().parent))
from hn_rules import make_hard_negatives


def load_rows(prompts_path, images_dir):
    files = {p.stem: p for p in Path(images_dir).rglob("*.png")}
    rows = []
    for line in open(prompts_path, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        text = r.get("prompt", r.get("prompts"))
        if not text or text == "hard_negative" or r["id"] not in files:
            continue
        rows.append({"id": r["id"], "prompt": text, "path": files[r["id"]]})
    return rows


class DS(Dataset):
    def __init__(self, rows, proc):
        self.rows, self.proc = rows, proc

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        img = Image.open(self.rows[i]["path"]).convert("RGB")
        return self.proc(images=img, return_tensors="pt")["pixel_values"][0], self.rows[i]["prompt"]


def collate(b):
    return torch.stack([x[0] for x in b]), [x[1] for x in b]


def set_trainable(model, last_n):
    """마지막 last_n 블록(비전/텍스트 각각) + 투영/정규화 층만 학습. 이름 패턴 기반이라 학습 파라미터 수를 출력해 확인."""
    for p in model.parameters():
        p.requires_grad = False
    layer_re = re.compile(r"^(.*?)\.layers\.(\d+)\.")
    maxidx = {}
    for n, _ in model.named_parameters():
        m = layer_re.match(n)
        if m:
            maxidx[m.group(1)] = max(maxidx.get(m.group(1), -1), int(m.group(2)))
    for n, p in model.named_parameters():
        m = layer_re.match(n)
        if m and int(m.group(2)) > maxidx[m.group(1)] - last_n:
            p.requires_grad = True
        elif re.search(r"(projection|post_layernorm|final_layer_norm|ln_post|ln_final|visual_projection|text_projection)", n):
            p.requires_grad = True
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[train] trainable params {tr / 1e6:.1f}M / {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M; towers={maxidx}")


def encode_text(model, tok, texts, dev, max_len, walk):
    ids = tok(texts, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")["input_ids"].to(dev)
    return F.normalize(model.get_text_features(ids, walk_short_pos=walk).float(), dim=-1)


@torch.no_grad()
def evaluate(model, tok, val_rows, val_px, dev, max_len, walk, amp):
    model.eval()
    with torch.autocast(dev.type, dtype=amp, enabled=dev.type == "cuda"):
        img = torch.cat([F.normalize(model.get_image_features(val_px[i:i + 64].to(dev)).float(), dim=-1)
                         for i in range(0, len(val_px), 64)])
        txt = torch.cat([encode_text(model, tok, [r["prompt"] for r in val_rows[i:i + 256]], dev, max_len, walk)
                         for i in range(0, len(val_rows), 256)])
    sim = txt @ img.T
    gt = sim.diag()
    rank = (sim > gt[:, None]).sum(1).cpu().numpy()  # 정답보다 점수가 높은 이미지 수
    r1 = float((rank == 0).mean())
    nd = float(np.where(rank < 5, 1.0 / np.log2(rank + 2), 0.0).mean())
    return {"recall@1": round(r1, 4), "ndcg@5": round(nd, 4), "score": round((r1 + nd) / 2, 4)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--prompts", required=True)
    ap.add_argument("--images", required=True)
    ap.add_argument("--repo", default="qihoo360/fg-clip-base")
    ap.add_argument("--out", required=True)
    ap.add_argument("--hn", type=int, default=0, help="프롬프트당 text HN 개수 (0=대조군)")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--last_n", type=int, default=4)
    ap.add_argument("--val", type=int, default=500)
    ap.add_argument("--val_ids", default="", help="번들 단위 val id 목록 (make_val_ids.py 출력). 없으면 무작위 분할(번들 누수 가능)")
    ap.add_argument("--long", action="store_true", help="긴 캡션 모드(max_length 248, walk_short_pos=False)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--workers", type=int, default=4)
    a = ap.parse_args()

    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    from transformers import AutoImageProcessor, AutoModelForCausalLM, AutoTokenizer
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = torch.bfloat16 if (dev.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
    max_len, walk = (248, False) if a.long else (77, True)

    rows = load_rows(a.prompts, a.images)
    if a.val_ids:
        ids = {l.strip() for l in open(a.val_ids) if l.strip()}
        norm = lambda x: str(int(x))
        ids = {norm(i) for i in ids}
        val_rows = [r for r in rows if norm(r["id"]) in ids]
        tr_rows = [r for r in rows if norm(r["id"]) not in ids]
        random.Random(a.seed).shuffle(tr_rows)
    else:
        random.Random(a.seed).shuffle(rows)
        val_rows, tr_rows = rows[:a.val], rows[a.val:]
    print(f"[data] train {len(tr_rows)} / val {len(val_rows)} | amp={amp} max_len={max_len} hn={a.hn}")

    model = AutoModelForCausalLM.from_pretrained(a.repo, trust_remote_code=True, torch_dtype=torch.float32).to(dev)
    tok = AutoTokenizer.from_pretrained(a.repo, trust_remote_code=True)
    proc = AutoImageProcessor.from_pretrained(a.repo, trust_remote_code=True)
    set_trainable(model, a.last_n)
    scale = model.logit_scale.exp().detach() if hasattr(model, "logit_scale") else torch.tensor(100.0, device=dev)
    scale = scale.clamp(max=100).to(dev)

    val_px = torch.stack([DS(val_rows, proc)[i][0] for i in range(len(val_rows))]).half()
    log = {"args": vars(a) | {"images": str(a.images)}, "epochs": []}
    log["zero_shot_val"] = evaluate(model, tok, val_rows, val_px, dev, max_len, walk, amp)
    print("[val] zero-shot", log["zero_shot_val"])

    dl = DataLoader(DS(tr_rows, proc), batch_size=a.bs, shuffle=True, num_workers=a.workers,
                    collate_fn=collate, drop_last=True, pin_memory=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=a.lr, weight_decay=0.05)
    total = a.epochs * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, int(0.1 * total))) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler(enabled=(dev.type == "cuda" and amp == torch.float16))
    rng = random.Random(a.seed)
    t0 = time.time()
    for ep in range(a.epochs):
        model.train()
        run = []
        for step, (px, prompts) in enumerate(dl):
            hn_txt = [t for p in prompts for _, t in make_hard_negatives(p, a.hn, rng)] if a.hn else []
            with torch.autocast(dev.type, dtype=amp, enabled=dev.type == "cuda"):
                img = F.normalize(model.get_image_features(px.to(dev)).float(), dim=-1)
                txt = encode_text(model, tok, prompts + hn_txt, dev, max_len, walk)
            b = len(prompts)
            tgt = torch.arange(b, device=dev)
            loss_i2t = F.cross_entropy(scale * img @ txt.T, tgt)       # HN 텍스트는 추가 negative 로만 들어감
            loss_t2i = F.cross_entropy(scale * txt[:b] @ img.T, tgt)
            loss = (loss_i2t + loss_t2i) / 2
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run.append(loss.item())
            if step % 20 == 0:
                print(f"[ep {ep}] step {step}/{len(dl)} loss {np.mean(run[-20:]):.4f}  {time.time() - t0:.0f}s", flush=True)
        v = evaluate(model, tok, val_rows, val_px, dev, max_len, walk, amp)
        print(f"[val] epoch {ep}", v)
        log["epochs"].append({"epoch": ep, "loss": float(np.mean(run)), "val": v})

    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    model.to(torch.bfloat16).save_pretrained(out, safe_serialization=True)
    tok.save_pretrained(out); proc.save_pretrained(out)
    (out / "train_log.json").write_text(json.dumps(log, indent=2, default=str))
    print("saved", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/submission/fgclip/model.py
"""FG-CLIP zero-shot 제출용 model.py (서버 계약: load_model / predict).

- load_model(model_dir, device): zip 루트에 번들된 FG-CLIP 가중치/원격코드를 로컬에서만 로드(네트워크 X).
- predict(model, queries, candidates, top_k=5): 8,000 쿼리 x 15,000 후보를 한 번에 받아
  {prompt_id: [image_id x top_k]} 반환. 이미지 임베딩은 이 호출 안에서 계산(캐시 없음).
fgclip_meta.json 으로 v1/v2 차이를 조절한다 (v2 경로는 실제 API 확인 전이라 미검증).
"""
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset


class _Bundle:
    def __init__(self, model, tokenizer, image_processor, meta, device):
        self.model, self.tokenizer, self.image_processor = model, tokenizer, image_processor
        self.meta, self.device = meta, device


def load_model(model_dir, device="cuda:0"):
    from transformers import AutoImageProcessor, AutoModelForCausalLM, AutoTokenizer

    d = str(Path(model_dir).resolve())
    meta_path = Path(d) / "fgclip_meta.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    model = AutoModelForCausalLM.from_pretrained(
        d, trust_remote_code=True, local_files_only=True, torch_dtype=torch.bfloat16
    )
    model = model.to(device).eval()
    tok = AutoTokenizer.from_pretrained(d, local_files_only=True, trust_remote_code=True)
    ip = AutoImageProcessor.from_pretrained(d, local_files_only=True, trust_remote_code=True)
    return _Bundle(model, tok, ip, meta, device)


class _ImgDS(Dataset):
    def __init__(self, paths, processor, patches=0):
        self.paths, self.processor, self.patches = paths, processor, patches

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        kw = {"max_num_patches": self.patches} if self.patches else {}
        out = self.processor(images=img, return_tensors="pt", **kw)
        return {k: v[0] for k, v in out.items()}


def _log(msg):
    print(f"[fgclip] {msg}", flush=True)


@torch.inference_mode()
def predict(model, queries, candidates, top_k=5):
    b, dev = model.model, model.device
    meta = model.meta
    max_len = int(meta.get("max_length", 77))
    walk = bool(meta.get("walk_short_pos", True))
    bs_img = int(meta.get("image_batch", 128))
    bs_txt = int(meta.get("text_batch", 512))

    t0 = time.time()
    paths = [c["image_path"] for c in candidates]
    ids = [str(c["image_id"]) for c in candidates]
    nw = min(8, os.cpu_count() or 2)
    v2 = int(meta.get("version", 1)) == 2
    patches = int(meta.get("max_num_patches", 256)) if v2 else 0
    dl = DataLoader(_ImgDS(paths, model.image_processor, patches), batch_size=bs_img,
                    num_workers=nw, pin_memory=True, prefetch_factor=4 if nw else None)
    feats = []
    for batch in dl:
        batch = {k: (v.to(dev, dtype=torch.bfloat16) if v.is_floating_point() else v.to(dev))
                 for k, v in batch.items()}
        f = b.get_image_features(**batch) if v2 else b.get_image_features(batch["pixel_values"])
        feats.append(torch.nn.functional.normalize(f.float(), dim=-1).to(torch.bfloat16))
    img = torch.cat(feats)
    t1 = time.time()
    _log(f"image encode {len(ids)} imgs: {t1 - t0:.1f}s")

    prompts = [q["prompt"] for q in queries]
    pids = [str(q["prompt_id"]) for q in queries]
    txt = []
    for s in range(0, len(prompts), bs_txt):
        enc = model.tokenizer([p.lower() for p in prompts[s:s + bs_txt]] if v2 else prompts[s:s + bs_txt],
                              padding="max_length", truncation=True,
                              max_length=max_len, return_tensors="pt").to(dev)
        if v2:
            f = b.get_text_features(**enc, walk_type=meta.get("walk_type", "short"))
        else:
            f = b.get_text_features(enc["input_ids"], walk_short_pos=walk)
        txt.append(torch.nn.functional.normalize(f.float(), dim=-1).to(torch.bfloat16))
    txt = torch.cat(txt)
    t2 = time.time()
    _log(f"text encode {len(pids)} prompts: {t2 - t1:.1f}s")

    out = {}
    for s in range(0, len(pids), 1000):
        sim = txt[s:s + 1000] @ img.T
        top = sim.float().topk(top_k, dim=1).indices.cpu().tolist()
        for pid, row in zip(pids[s:s + 1000], top):
            out[pid] = [ids[j] for j in row]
    _log(f"search: {time.time() - t2:.1f}s | peak VRAM {torch.cuda.max_memory_allocated() / 1e9:.1f}GB"
         if torch.cuda.is_available() else "search done")
    return out

In [ ]:
%%writefile /content/work/scripts/build_fgclip_submission.py
"""FG-CLIP HF 스냅샷 -> 서버 형식 submission.zip 빌드 + 서버 규칙 검증.

사용 (Colab/Mac, HF 접근 가능한 곳):
  pip install huggingface_hub
  python scripts/build_fgclip_submission.py --repo qihoo360/fg-clip-large --out build/fgclip_large.zip --walk-short-pos 1
레포 id는 HF에서 정확한 이름을 확인해 넣을 것 (v2는 --walk-short-pos 0 등 API 확인 필요).
zip 루트에 model.py + *.safetensors + config/tokenizer/preprocessor + 원격코드(*.py) 포함.
"""
import argparse, json, shutil, sys, zipfile
from pathlib import Path

HERE = Path(__file__).resolve().parent.parent
SKIP_SUFFIX = {".bin", ".h5", ".msgpack", ".onnx", ".gguf"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--walk-short-pos", type=int, default=1)
    ap.add_argument("--max-length", type=int, default=77)
    ap.add_argument("--version", type=int, default=1, help="1=fg-clip, 2=fg-clip2")
    ap.add_argument("--walk-type", default="short", help="v2: short(max_len 64) / long(196)")
    ap.add_argument("--max-num-patches", type=int, default=256, help="v2 이미지 패치 수 (128/256/576/784/1024)")
    ap.add_argument("--req", action="append", default=[], help="requirements.txt 줄 추가 (여러 번 가능, 예: --req einops==0.8.1)")
    ap.add_argument("--transformers-pin", default="", help="예: transformers==4.46.3 (모델 카드 권장 버전)")
    a = ap.parse_args()

    if Path(a.repo).is_dir():   # 파인튜닝 결과 폴더 (save_pretrained 출력)
        snap = Path(a.repo)
    else:
        from huggingface_hub import snapshot_download
        snap = Path(snapshot_download(a.repo))
    stage = Path(a.out).with_suffix("")
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    for p in snap.iterdir():
        if p.is_file() and p.name != "model.py" and p.suffix not in SKIP_SUFFIX and not p.name.startswith("."):
            shutil.copy(p.resolve(), stage / p.name)  # symlink 해제 복사
    # 원격코드가 model.py 를 덮지 않도록: 원본 repo의 modeling 파일은 그대로 두고 제출 model.py만 추가
    shutil.copy(HERE / "submission/fgclip/model.py", stage / "model.py")
    (stage / "fgclip_meta.json").write_text(json.dumps(
        {"repo": a.repo, "version": a.version, "walk_short_pos": bool(a.walk_short_pos),
         "max_length": a.max_length, "walk_type": a.walk_type, "max_num_patches": a.max_num_patches}))
    reqs = ([a.transformers_pin] if a.transformers_pin else []) + a.req
    if reqs:
        (stage / "requirements.txt").write_text("\n".join(reqs) + "\n")

    if not list(stage.glob("*.safetensors")):
        sys.exit("safetensors 가중치가 없음 (repo가 .bin만 제공?) -> safetensors로 변환 필요")
    out = Path(a.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for p in sorted(stage.iterdir()):
            z.write(p, p.name)
    print(f"zip: {out} ({out.stat().st_size / 1e9:.2f} GB)")
    shutil.rmtree(stage, ignore_errors=True)  # 스테이징 복사본 삭제 (디스크 절약)

    # 서버 규칙 검증 (2026-HAIKUTHON 의 evaluation/model_loader.py 재사용)
    srv = Path(__import__("os").environ.get("HAIKUTHON_SERVER_REPO", ""))
    if (srv / "evaluation/model_loader.py").exists():
        sys.path.insert(0, str(srv))
        from evaluation.model_loader import prepare_submission
        import tempfile
        with tempfile.TemporaryDirectory() as t:
            prepare_submission(out, Path(t) / "x")
        print("서버 prepare_submission 검증 통과")
    else:
        print("HAIKUTHON_SERVER_REPO 미설정 -> 서버 규칙 검증 생략 (2026-HAIKUTHON 경로를 지정하면 검증)")


if __name__ == "__main__":
    main()

In [ ]:
# hn_rules 동작 확인: 실제 train 프롬프트 3개에 대한 hard negative 예시
import sys, random, json; sys.path.insert(0, "/content/work")
from hn_rules import make_hard_negatives
rows = [json.loads(l) for l in open(PROMPTS)]
rows = [r for r in rows if r.get("prompts", r.get("prompt")) not in (None, "hard_negative")][:3]
for r in rows:
    t = r.get("prompts", r.get("prompt")); print("ORIG:", t)
    for kind, h in make_hard_negatives(t, 3, random.Random(0)): print(f"  [{kind}]", h)
    print()

## 6. 대조군 학습 (`--hn 0`)

In [ ]:
VAL_ARG = f'--val_ids "{VAL_IDS_PATH}"' if VAL_IDS_PATH else ""
!python /content/work/train_fgclip_hn.py --prompts "{PROMPTS}" --images "{IMAGES}" --repo {REPO} \
    --hn 0 --epochs {EPOCHS} --bs {BS} --lr {LR} --last_n {LAST_N} --seed {SEED} {VAL_ARG} --out "{OUT_DIR}/ctrl"

## 7. text hard negative 학습 (`--hn HN`)

In [ ]:
!python /content/work/train_fgclip_hn.py --prompts "{PROMPTS}" --images "{IMAGES}" --repo {REPO} \
    --hn {HN} --epochs {EPOCHS} --bs {BS} --lr {LR} --last_n {LAST_N} --seed {SEED} {VAL_ARG} --out "{OUT_DIR}/hn{HN}"

## 8. 결과 비교 (val 500, 상대 비교용)

In [ ]:
import pandas as pd
rows = []
for name in ["ctrl", f"hn{HN}"]:
    log = json.load(open(f"{OUT_DIR}/{name}/train_log.json"))
    z = log["zero_shot_val"]; rows.append({"run": name, "stage": "zero-shot", **z})
    for e in log["epochs"]:
        rows.append({"run": name, "stage": f"epoch {e['epoch']}", **e["val"]})
df = pd.DataFrame(rows); display(df)
print("최종 에폭 score:", {n: df[(df.run==n)].iloc[-1]["score"] for n in ["ctrl", f"hn{HN}"]})

## 9. 서버 제출용 zip 생성 (Drive 에 저장)

파인튜닝 폴더에 원격코드(`*.py`)가 빠졌을 수 있어 원본 repo 에서 없는 파일만 복사한 뒤 zip 을 만든다.
만들어진 zip 은 Drive 에서 Mac 으로 받아 `modal volume put jp-fgclip-zip ...` → `modal run ... --zip <이름>` 으로 L4 에서 평가한다.

In [ ]:
import shutil
from huggingface_hub import snapshot_download
snap = snapshot_download(REPO, allow_patterns=["*.py", "*.json", "*.txt"])
os.makedirs(f"{OUT_DIR}/zips", exist_ok=True)

for name in ["ctrl", f"hn{HN}"]:
    d = f"{OUT_DIR}/{name}"
    for f in glob.glob(snap + "/*.py"):
        dst = os.path.join(d, os.path.basename(f))
        if not os.path.exists(dst): shutil.copy(f, dst)
    !python /content/work/scripts/build_fgclip_submission.py --repo "{d}" --out "{OUT_DIR}/zips/fgclip_ft_{name}.zip" \
        --transformers-pin {TRANSFORMERS_PIN} --req einops==0.8.1 --req torchvision==0.29.2
!ls -lh "{OUT_DIR}/zips"